# 🔬 Model Architecture Analysis

This notebook analyzes the model architecture and parameters.

In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import torch.nn as nn
from collections import defaultdict

from src.models import Gemma3Model, GEMMA3_CONFIG_270M

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

## 1. Model Architecture

In [ ]:
# Initialize model
model = Gemma3Model(GEMMA3_CONFIG_270M)
print(f"Model loaded with {GEMMA3_CONFIG_270M['emb_dim']} embedding dimension")
print(f"Number of layers: {GEMMA3_CONFIG_270M['n_layers']}")
print(f"Number of attention heads: {GEMMA3_CONFIG_270M['n_heads']}")
print(f"KV Groups: {GEMMA3_CONFIG_270M['n_kv_groups']}")
print(f"Context length: {GEMMA3_CONFIG_270M['context_length']}")
print(f"Sliding window: {GEMMA3_CONFIG_270M['sliding_window']}")

In [ ]:
# Display layer types
layer_types = GEMMA3_CONFIG_270M['layer_types']
print("Layer Types:")
for i, layer_type in enumerate(layer_types):
    symbol = "🟦" if layer_type == "sliding_attention" else "🟥"
    print(f"Layer {i:2d}: {symbol} {layer_type}")

# Count layer types
from collections import Counter
layer_counts = Counter(layer_types)
print("\nLayer Distribution:")
for layer_type, count in layer_counts.items():
    print(f"  {layer_type}: {count} layers ({count/len(layer_types)*100:.1f}%)")

## 2. Parameter Statistics

In [ ]:
def count_parameters(model):
    """Count parameters by category."""
    param_stats = defaultdict(int)
    total_params = 0
    
    for name, param in model.named_parameters():
        if param.requires_grad:
            num_params = param.numel()
            total_params += num_params
            
            # Categorize
            if 'embedding' in name.lower() or 'tok_emb' in name.lower():
                param_stats['Embeddings'] += num_params
            elif 'attn' in name.lower() or 'attention' in name.lower():
                param_stats['Attention'] += num_params
            elif 'ff' in name.lower() or 'feedforward' in name.lower():
                param_stats['FeedForward'] += num_params
            elif 'norm' in name.lower():
                param_stats['Normalization'] += num_params
            elif 'out_head' in name.lower() or 'lm_head' in name.lower():
                param_stats['Output'] += num_params
            else:
                param_stats['Other'] += num_params
    
    return total_params, param_stats

total_params, param_stats = count_parameters(model)

print(f"Total parameters: {total_params:,}")
print(f"Total parameters (millions): {total_params/1e6:.2f}M")
print("\nParameter Distribution:")
for category, count in param_stats.items():
    percentage = count / total_params * 100
    print(f"  {category:15s}: {count:>10,} ({percentage:5.1f}%)")

In [ ]:
# Visualize parameter distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart
categories = list(param_stats.keys())
values = list(param_stats.values())
colors = sns.color_palette("pastel")

ax1.pie(values, labels=categories, autopct='%1.1f%%', colors=colors, startangle=90)
ax1.set_title('Parameter Distribution by Category')

# Bar chart
ax2.barh(categories, [v/1e6 for v in values], color=colors)
ax2.set_xlabel('Parameters (Millions)')
ax2.set_title('Parameter Counts by Category')

plt.tight_layout()
plt.show()

## 3. Memory Usage

In [ ]:
def estimate_memory_usage(model, batch_size=1, seq_length=128):
    """Estimate memory usage."""
    param_size = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.numel() * b.element_size() for b in model.buffers())
    
    # Estimate activation memory
    hidden_size = model.cfg['emb_dim']
    n_layers = model.cfg['n_layers']
    
    # Rough estimation
    activation_size = (batch_size * seq_length * hidden_size * n_layers * 4)  # 4 bytes per float
    
    total_memory = param_size + buffer_size + activation_size
    
    return {
        'parameters': param_size,
        'buffers': buffer_size,
        'activations': activation_size,
        'total': total_memory
    }

memory_estimates = estimate_memory_usage(model)

print("Memory Usage Estimates:")
print(f"  Parameters:        {memory_estimates['parameters']/1e9:.3f} GB")
print(f"  Buffers:           {memory_estimates['buffers']/1e9:.3f} GB")
print(f"  Activations:       {memory_estimates['activations']/1e9:.3f} GB")
print(f"  Total:             {memory_estimates['total']/1e9:.3f} GB")
print(f"  Total (MiB):       {memory_estimates['total']/1024**3:.3f} GiB")

## 4. Attention Patterns Analysis

In [ ]:
def analyze_attention_masks(seq_len=32):
    """Analyze attention mask patterns."""
    mask_global, mask_local = model._create_masks(seq_len, 'cpu')
    
    # Count masked positions
    global_masked = mask_global.sum().item()
    local_masked = mask_local.sum().item()
    total_positions = seq_len * seq_len
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Global mask
    axes[0].imshow(mask_global.numpy(), cmap='RdBu_r', aspect='auto')
    axes[0].set_title(f'Global Attention Mask\nMasked: {global_masked/total_positions*100:.1f}%')
    axes[0].set_xlabel('Key Position')
    axes[0].set_ylabel('Query Position')
    
    # Local mask
    axes[1].imshow(mask_local.numpy(), cmap='RdBu_r', aspect='auto')
    axes[1].set_title(f'Local (Sliding) Mask\nMasked: {local_masked/total_positions*100:.1f}%')
    axes[1].set_xlabel('Key Position')
    axes[1].set_ylabel('Query Position')
    
    # Difference
    diff = mask_local.float() - mask_global.float()
    axes[2].imshow(diff.numpy(), cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
    axes[2].set_title('Difference (Local - Global)')
    axes[2].set_xlabel('Key Position')
    axes[2].set_ylabel('Query Position')
    
    plt.tight_layout()
    plt.show()

analyze_attention_masks()

## 5. Model Speed Analysis

In [ ]:
import time

def benchmark_model(model, seq_lengths=[32, 64, 128, 256, 512], device='cpu'):
    """Benchmark model inference speed."""
    model = model.to(device)
    model.eval()
    
    results = []
    
    for seq_len in seq_lengths:
        # Create input
        input_ids = torch.randint(0, 1000, (1, seq_len)).to(device)
        
        # Warmup
        for _ in range(3):
            with torch.no_grad():
                _ = model(input_ids)
        
        # Measure time
        torch.cuda.synchronize() if device == 'cuda' else None
        start_time = time.time()
        
        with torch.no_grad():
            for _ in range(10):
                _ = model(input_ids)
        
        torch.cuda.synchronize() if device == 'cuda' else None
        end_time = time.time()
        
        avg_time = (end_time - start_time) / 10
        results.append({'seq_len': seq_len, 'time': avg_time, 'tokens_per_sec': seq_len / avg_time})
    
    return results

# Benchmark on CPU (or GPU if available)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Benchmarking on {device.upper()}")

benchmark_results = benchmark_model(model, device=device)
df = pd.DataFrame(benchmark_results)
df

# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(df['seq_len'], df['time'], marker='o', linewidth=2)
ax1.set_xlabel('Sequence Length')
ax1.set_ylabel('Inference Time (seconds)')
ax1.set_title('Inference Time vs Sequence Length')
ax1.grid(True, alpha=0.3)

ax2.plot(df['seq_len'], df['tokens_per_sec'], marker='o', linewidth=2, color='green')
ax2.set_xlabel('Sequence Length')
ax2.set_ylabel('Tokens per Second')
ax2.set_title('Throughput vs Sequence Length')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Save Analysis Results

In [ ]:
# Save analysis results
analysis_results = {
    'config': GEMMA3_CONFIG_270M,
    'total_parameters': total_params,
    'parameter_distribution': dict(param_stats),
    'memory_estimates': memory_estimates,
    'benchmark_results': benchmark_results
}

import json
with open('../logs/model_analysis.json', 'w') as f:
    json.dump(analysis_results, f, indent=2, default=lambda x: str(x) if isinstance(x, torch.dtype) else x)

print("Analysis results saved to logs/model_analysis.json")